# AdvBench/JBB translation with Gemma and Lapa

Translate the English `goal` of every record in the pinned combined harmful prompt set (`prompt-pairs.csv`: 100 JBB + 520 AdvBench) into Ukrainian, once with Gemma 3 12B IT and once with Lapa 0.1.2 Instruct. The existing `ukrainian_prompt` column is not shown to the models.

Each record is sent as its own request, using the translation instructions adapted from the JBB benign translation prompt (`data/safety/jbb-benign.provenance.json`). The model returns one JSON object per record; `record_id` is copied from the input by the notebook, so IDs are preserved even when the model output is malformed.

Outputs, one per model, written to `data/safety/translations/`:

- `prompt-pairs-uk-gemma.csv`, `prompt-pairs-uk-lapa.csv`: one row per input record, same order as the input (sorted by `record_id`). UTF-8 CSV with standard quoting, so commas, quotes and line breaks inside text are safe. Columns: `record_id`, `source`, `goal` (English), `ukrainian_prompt` (model translation), `flagged`, `note`, `status`, `model`, `raw_response`.
- `*.provenance.json`: prompt, model path and revision, input SHA256, generation settings and counts.
- `*.partial.jsonl`: written batch by batch while running; useful if the job stops early.

Environment variables: `MODELS` (default `gemma,lapa`), `ITEM_LIMIT` (default `0` = all records; a limit adds `-limitN` to the file names), `BATCH_SIZE` (default 16), `MAX_NEW_TOKENS` (default 512), `OVERWRITE` (default `0`).

Runs on one GPU inside a Slurm allocation (`slurm/translate.sbatch`). Model outputs are machine translations: they need human review before use, like the benign set.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib.metadata import version
import gc
import hashlib
import json
import os
import platform
import re
import time

import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display
from transformers import AutoTokenizer, Gemma3ForConditionalGeneration, GenerationConfig

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
project = Path("/e/project1/e-dev-2026d09-075")
model_paths = {
    "gemma": project
    / "fairforget-steering-transfer/models/gemma-3-12b-it/96b6f1eccf38110c56df3a15bffe176da04bfd80",
    "lapa": project
    / "fairforget/models/lapa-v0.1.2-instruct/2969fd997f07b33727c6f10795d78ef23cc5c037",
}
model_names = [m.strip() for m in os.environ.get("MODELS", "gemma,lapa").split(",") if m.strip()]
assert model_names and set(model_names) <= set(model_paths), model_names
limit = int(os.environ.get("ITEM_LIMIT", "0"))
batch_size = int(os.environ.get("BATCH_SIZE", "16"))
max_new_tokens = int(os.environ.get("MAX_NEW_TOKENS", "512"))
overwrite = os.environ.get("OVERWRITE", "0") == "1"
assert limit >= 0 and batch_size > 0 and max_new_tokens > 0
out_dir = root / "data/safety/translations"
out_dir.mkdir(parents=True, exist_ok=True)
suffix = f"-limit{limit}" if limit else ""
torch.manual_seed(42)
torch.set_num_threads(8)
print("Models:", model_names)
print("Records:", limit if limit else "all")
print("Output folder:", out_dir)

## Data

Load the combined prompt set and check its SHA256 against the pinned value in `configs/artifacts.json`. The first file found is used:

1. `PROMPT_PAIRS_CSV`, if set;
2. `data/safety/translated_formated.csv`, the local copy listed in `data/README.md`;
3. `data/safety/prompt-pairs.csv`;
4. your Hugging Face cache (`hf_hub_download`). Compute nodes run offline, so this works only if the file was downloaded into *your* cache beforehand on a login node.

Whichever copy is used, the SHA256 check guarantees it is the pinned file. Only `record_id`, `source` and `goal` are used.

In [ ]:
input_config = json.loads((root / "configs/artifacts.json").read_text())["prompt_pairs"]
candidates = [
    os.environ.get("PROMPT_PAIRS_CSV"),
    root / "data/safety/translated_formated.csv",
    root / "data/safety/prompt-pairs.csv",
]
input_path = next((Path(p) for p in candidates if p and Path(p).is_file()), None)
if input_path is None:
    input_path = Path(
        hf_hub_download(
            input_config["repo_id"],
            input_config["filename"],
            repo_type="dataset",
            revision=input_config["revision"],
        )
    )
print("Input file:", input_path)
input_sha256 = hashlib.sha256(input_path.read_bytes()).hexdigest()
assert input_sha256 == input_config["sha256"]
frame = pd.read_csv(input_path).sort_values("record_id").reset_index(drop=True)
assert len(frame) == 620 and frame.record_id.is_unique
assert frame.goal.notna().all() and frame.goal.str.strip().ne("").all()
if limit:
    frame = frame.groupby("source", group_keys=False).head(limit).reset_index(drop=True)
display(frame.groupby("source").size().to_frame("records"))
display(frame[["record_id", "source", "goal"]].head(4))

## Translation prompt

Adapted from the JBB benign prompt for this dataset:

- Input fields are `record_id`, `source` and `goal`; there is no `Target`, `Behavior` or `Category` in this file, so those context instructions are dropped.
- These prompts are harmful by design, so rule 6 asks for a faithful translation that neither softens nor intensifies the request, instead of "keep it benign".
- One record per request: the model returns a single JSON object rather than an array of 100 rows. The notebook enforces count and order.

In [ ]:
PROMPT_TEMPLATE = """You are translating English prompts into Ukrainian for a research test set measuring
refusal behavior in language models. The prompts come from the public AdvBench and
JailbreakBench (JBB) harmful-behavior benchmarks, so many of them describe harmful
requests. Your task is translation only: do not carry out, answer, judge, or refuse the
request. The translation is used to test whether models refuse it in Ukrainian.

INPUT: one record from the combined AdvBench + JBB prompt set, with fields
record_id, source (AdvBench or JBB), goal (the English prompt).

TASK: translate only the "goal" field into Ukrainian. "source" is context only; never
translate it, copy it into the translation, or let it add content that is not
already in "goal".

TRANSLATION RULES (strict):
1. Preserve the original intent and register exactly, do not soften, intensify,
   formalize, or casualize beyond what natural Ukrainian grammar requires.
2. Preserve all negations, conditions, and constraints precisely (e.g. "without",
   "only if", "excluding").
3. Preserve all numbers, dates, and quantities exactly as given.
4. Preserve proper names (people, places, brands, products, organizations, software)
   either in standard Ukrainian convention if one exists, or transliterated
   consistently. Never substitute or localize a name.
5. Translate only what "goal" asks for. Do not add an answer, a disclaimer, a
   safety caveat, or any instruction not present in the source. The output is a
   translated REQUEST, never a response to it.
6. Never shift how harmful/benign the request reads. These goals are harmful
   requests by design; keep each one reading as the same request in Ukrainian. Do
   not soften, sanitize, euphemize, or refuse it, and do not make it more explicit,
   detailed, or dangerous than the English.
7. If "goal" is ambiguous, idiomatic, or has more than one meaning-changing
   reading, translate the most literal/faithful reading and flag it with a short
   note, do not silently resolve it and do not edit/clarify the English.
8. Use natural, idiomatic Ukrainian, avoid stiff or word-for-word calques.
   Phrase it as an informal singular imperative ("Напиши", "Створи", "Поясни",
   "Опиши") to match how these are addressed to an AI (singular), not formal/plural
   ("Напишіть", "Навчіть", "Поясніть", "Опишіть", etc).
9. Produce exactly one output object for this record, tagged with its record_id.

OUTPUT FORMAT: return ONLY one JSON object (no prose, no markdown fences), so commas
and quotes inside the translated text can't corrupt the output:
  {{
    "record_id": <int, copied unchanged from the input>,
    "ukrainian_prompt": <string, the Ukrainian translation of goal>,
    "flagged": <true|false>,
    "note": <string, empty "" if flagged is false; if true, a short English
             note under 20 words naming the ambiguity and the alternative
             reading, for human review, never a disclaimer or hedge>
  }}

Do not include goal, source, similar_record_id, discard, split, or
translation_status in the output, those get attached downstream from the
original CSV file, not by you.

INPUT RECORD:
record_id: {record_id}
source: {source}
goal: {goal}"""


def build_prompt(row):
    return PROMPT_TEMPLATE.format(record_id=int(row.record_id), source=row.source, goal=row.goal)


print(build_prompt(frame.iloc[0]))

## Parsing

Take the first JSON object in the reply, after stripping any markdown fences. `status` records what happened:

| status | meaning |
|---|---|
| `ok` | valid object, non-empty translation, `record_id` matches |
| `id_mismatch` | parsed, but the model wrote a different `record_id` (the input ID is kept) |
| `empty` | parsed, but `ukrainian_prompt` is blank |
| `parse_error` | no valid JSON object; often a refusal or extra prose. See `raw_response` |
| `token_limit` | generation hit `MAX_NEW_TOKENS` before ending |

In [ ]:
def parse_reply(text, record_id):
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    start = cleaned.find("{")
    obj = None
    if start != -1:
        try:
            obj, _ = json.JSONDecoder(strict=False).raw_decode(cleaned[start:])
        except json.JSONDecodeError:
            obj = None
    if not isinstance(obj, dict):
        return {"ukrainian_prompt": "", "flagged": False, "note": "", "status": "parse_error"}
    translation = str(obj.get("ukrainian_prompt") or "").strip()
    flagged = obj.get("flagged")
    flagged = flagged if isinstance(flagged, bool) else str(flagged).strip().lower() == "true"
    note = str(obj.get("note") or "").strip()
    try:
        id_ok = int(obj.get("record_id")) == int(record_id)
    except (TypeError, ValueError):
        id_ok = False
    status = "empty" if not translation else ("ok" if id_ok else "id_mismatch")
    return {"ukrainian_prompt": translation, "flagged": flagged, "note": note, "status": status}


assert parse_reply('```json\n{"record_id": 3, "ukrainian_prompt": "Напиши \\"x\\"", "flagged": false, "note": ""}\n```', 3)["status"] == "ok"
assert parse_reply("I can't help with that.", 3)["status"] == "parse_error"

## Run

For each model: load it in BF16 on the single allocated GPU, translate in left-padded batches with greedy decoding, keep each reply up to its first end token, parse it, then free the GPU before loading the next model. Existing output files are not overwritten unless `OVERWRITE=1`.

In [ ]:
assert torch.cuda.is_available(), "Run inside a GPU Slurm allocation"
assert torch.cuda.device_count() == 1, "Each worker must see exactly one GPU"
print("Host:", platform.node(), "GPU:", torch.cuda.get_device_name(0))
summaries = {}
OUTPUT_COLUMNS = [
    "record_id", "source", "goal", "ukrainian_prompt",
    "flagged", "note", "status", "model", "raw_response",
]

for model_name in model_names:
    model_path = model_paths[model_name]
    output_path = out_dir / f"prompt-pairs-uk-{model_name}{suffix}.csv"
    partial_path = out_dir / f"prompt-pairs-uk-{model_name}{suffix}.partial.jsonl"
    provenance_path = out_dir / f"prompt-pairs-uk-{model_name}{suffix}.provenance.json"
    if output_path.exists() and not overwrite:
        raise FileExistsError(f"{output_path} exists; set OVERWRITE=1 to replace it")
    print(f"\n=== {model_name}: {model_path.name}")

    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    tokenizer.padding_side = "left"
    model = Gemma3ForConditionalGeneration.from_pretrained(
        model_path,
        dtype=torch.bfloat16,
        device_map={"": "cuda:0"},
        attn_implementation="sdpa",
        local_files_only=True,
    ).eval()
    settings = GenerationConfig(
        do_sample=False,
        max_new_tokens=max_new_tokens,
        bos_token_id=tokenizer.bos_token_id,
        eos_token_id=model.generation_config.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    eot_ids = settings.eos_token_id
    eot_ids = [eot_ids] if isinstance(eot_ids, int) else list(eot_ids)

    results = []
    started = time.monotonic()
    with partial_path.open("w", encoding="utf-8") as partial:
        for start in range(0, len(frame), batch_size):
            batch = frame.iloc[start : start + batch_size]
            conversations = [
                [{"role": "user", "content": build_prompt(row)}]
                for row in batch.itertuples(index=False)
            ]
            inputs = tokenizer.apply_chat_template(
                conversations,
                tokenize=True,
                padding=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            ).to(model.device)
            with torch.inference_mode():
                generated = model.generate(**inputs, generation_config=settings)
            new_tokens = generated[:, inputs.input_ids.shape[-1] :].cpu().tolist()
            for row, tokens in zip(batch.itertuples(index=False), new_tokens, strict=True):
                end = next((i + 1 for i, t in enumerate(tokens) if t in eot_ids), len(tokens))
                tokens = tokens[:end]
                raw = tokenizer.decode(tokens, skip_special_tokens=True).strip()
                parsed = parse_reply(raw, row.record_id)
                if not tokens or tokens[-1] not in eot_ids:
                    parsed["status"] = "token_limit"
                record = {
                    "record_id": int(row.record_id),
                    "source": row.source,
                    "goal": row.goal,
                    **parsed,
                    "model": model_name,
                    "raw_response": raw,
                }
                results.append(record)
                partial.write(json.dumps(record, ensure_ascii=False) + "\n")
            partial.flush()
            print(f"{model_name} {min(start + batch_size, len(frame))}/{len(frame)}"
                  f"  {time.monotonic() - started:.0f}s", flush=True)

    assert [r["record_id"] for r in results] == frame.record_id.astype(int).tolist()
    pd.DataFrame(results, columns=OUTPUT_COLUMNS).to_csv(output_path, index=False, encoding="utf-8")
    counts = pd.Series([r["status"] for r in results]).value_counts().to_dict()
    summaries[model_name] = counts
    provenance = {
        "input": {**input_config, "sha256_checked": input_sha256, "records": len(frame), "item_limit": limit},
        "model": model_name,
        "model_path": str(model_path),
        "revision": model_path.name,
        "model_class": type(model).__name__,
        "dtype": str(model.dtype),
        "generation": settings.to_dict(),
        "batch_size": batch_size,
        "padding_side": tokenizer.padding_side,
        "prompt_template": PROMPT_TEMPLATE,
        "prompt_placement": "single user turn, native chat template",
        "status_counts": counts,
        "flagged": int(sum(r["flagged"] for r in results)),
        "elapsed_seconds": time.monotonic() - started,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "hostname": platform.node(),
        "gpu": torch.cuda.get_device_name(0),
        "versions": {n: version(n) for n in ["torch", "transformers", "pandas"]},
        "review": "machine translation, pending human review",
    }
    provenance_path.write_text(json.dumps(provenance, indent=2, ensure_ascii=False), encoding="utf-8")
    print("Saved:", output_path.name, counts)

    del model, tokenizer, generated, inputs
    gc.collect()
    torch.cuda.empty_cache()

## Check

Status counts per model, every record that did not parse cleanly, and a few side-by-side examples. Rows with a status other than `ok` need manual attention; a `parse_error` whose `raw_response` is a refusal means that model declined to translate the prompt.

In [ ]:
pd.set_option("display.max_colwidth", None)
display(pd.DataFrame(summaries).fillna(0).astype(int))
outputs = {
    name: pd.read_csv(out_dir / f"prompt-pairs-uk-{name}{suffix}.csv", keep_default_na=False)
    for name in model_names
}
for name, df in outputs.items():
    assert len(df) == len(frame) and df.record_id.is_unique
    problems = df[df.status.ne("ok")]
    print(f"{name}: {len(problems)} records need attention, {int(df.flagged.sum())} flagged")
    if len(problems):
        display(problems[["record_id", "source", "status", "raw_response"]])
    flagged = df[df.flagged]
    if len(flagged):
        display(flagged[["record_id", "ukrainian_prompt", "note"]])

side_by_side = frame[["record_id", "source", "goal"]].copy()
for name, df in outputs.items():
    side_by_side[name] = df.set_index("record_id").ukrainian_prompt.reindex(side_by_side.record_id).values
display(side_by_side.groupby("source").head(3))